In [0]:
#Load the required tables
from pyspark.sql import functions as F

membership_df = spark.table("segment_membership")
consent_df = spark.table("consent")
order_df = spark.table("order_master")
order_line_df = spark.table("order_line")
recommendation_df = spark.table("customer_recommendation")
outcome_df = spark.table("action_outcome")

print("All tables loaded successfully!")

In [0]:
#Create a separate ground-truth table
ground_truth_data = [
    ("ISSUE-001", "CONFLICTING_SEGMENTS",
     "One customer is assigned to two exclusive segments",
     "HIGH"),

    ("ISSUE-002", "CONSENT_VIOLATION",
     "Marketing action is assigned to a customer with withdrawn consent",
     "CRITICAL"),

    ("ISSUE-003", "RETURN_NOT_NETTED",
     "Returned order spending is incorrectly included in monetary value",
     "HIGH"),

    ("ISSUE-004", "WRONG_ACTION_ATTRIBUTION",
     "Outcome is connected to the wrong recommended action",
     "HIGH")
]

ground_truth_df = spark.createDataFrame(
    ground_truth_data,
    ["issue_id", "issue_type", "description", "severity"]
)

display(ground_truth_df)

In [0]:

ground_truth_df.write.format("delta").mode("overwrite").saveAsTable(
    "ground_truth"
)

In [0]:
#Test conflicting customer segments
test_customer = membership_df.select("customer_id").first()["customer_id"]

existing_membership = membership_df.filter(
    F.col("customer_id") == test_customer
)

conflicting_membership = (
    existing_membership
    .withColumn("membership_id", F.concat(F.lit("TEST-MEM-"), F.col("customer_id")))
    .withColumn("segment_id", F.lit("SEG-2026-000005"))
    .withColumn("segment_name", F.lit("Hibernating"))
    .withColumn("reason_text", F.lit("Deliberately injected conflicting segment"))
)

membership_test_df = membership_df.unionByName(conflicting_membership)

membership_issues_df = (
    membership_test_df
    .groupBy("customer_id")
    .agg(F.countDistinct("segment_id").alias("segment_count"))
    .filter(F.col("segment_count") > 1)
    .withColumn("issue_type", F.lit("CONFLICTING_SEGMENTS"))
)

display(membership_issues_df)

In [0]:
#Test marketing consent violation
withdrawn_customer = (
    consent_df
    .filter(F.col("consent_status") == "WITHDRAWN")
    .select("customer_id")
    .first()
)

In [0]:
consent_test_df = (
    recommendation_df
    .filter(F.col("customer_id") == withdrawn_customer["customer_id"])
    .limit(1)
    .withColumn("recommended_action", F.lit("LOYALTY_OFFER"))
    .withColumn("action_type", F.lit("MARKETING"))
    .withColumn("consent_status", F.lit("WITHDRAWN"))
)

display(consent_test_df)

In [0]:
consent_issues_df = consent_test_df.filter(
    (F.col("action_type") == "MARKETING") &
    (F.col("consent_status") != "GRANTED")
).withColumn(
    "issue_type",
    F.lit("CONSENT_VIOLATION")
)

display(consent_issues_df)

In [0]:
#Test return not netted from spending
returned_order_df = (
    order_df
    .filter(F.col("order_status") == "RETURNED")
)

display(returned_order_df)

In [0]:
returned_spend_df = (
    returned_order_df
    .join(order_line_df, "order_id", "inner")
    .groupBy("order_id", "customer_id")
    .agg(F.sum("line_total").alias("returned_amount"))
)

display(returned_spend_df)

In [0]:
return_issues_df = returned_spend_df.withColumn(
    "issue_type",
    F.lit("RETURN_NOT_NETTED")
)

display(return_issues_df)

In [0]:
#Test wrong action attribution
wrong_action_test_df = (
    outcome_df
    .filter(F.col("action_id").isNotNull())
    .limit(1)
    .withColumn("action_id", F.lit("ACTN-2026-999999"))
    .withColumn(
        "issue_type",
        F.lit("WRONG_ACTION_ATTRIBUTION")
    )
)

display(wrong_action_test_df)

In [0]:
action_catalog_df = spark.table("action_catalog")

wrong_action_issues_df = (
    wrong_action_test_df
    .join(
        action_catalog_df.select("action_id"),
        "action_id",
        "left_anti"
    )
)

display(wrong_action_issues_df)

In [0]:
#Save test results separately
membership_issues_df.write.format("delta").mode("overwrite").saveAsTable(
    "test_conflicting_segments"
)

consent_issues_df.write.format("delta").mode("overwrite").saveAsTable(
    "test_consent_violations"
)

return_issues_df.write.format("delta").mode("overwrite").saveAsTable(
    "test_return_issues"
)

wrong_action_issues_df.write.format("delta").mode("overwrite").saveAsTable(
    "test_wrong_action_attribution"
)

print("All edge-case test tables created successfully!")